# NISAR ISCE3 PGE

Papermill-driven notebook PGE. Runs under the **`isce3_src`** kernel so that `nisar`/`isce3` (and the `stage_dem` subprocess spawned by the localizer) resolve.

Steps:
1. Write netrc credentials, put `nisar_products` on the path, point at the track/frame DB.
2. Localize the S3 runconfig (download inputs, stage DEM) to local paths.
3. Dispatch the NISAR SAS workflow for the runconfig product type.
4. Stage out every deliverable HDF5: rename to its full NISAR granule name (read from HDF5 metadata) and move each into a sibling directory named for the granule, alongside a copy of the runconfig used. A combined InSAR runconfig (e.g. `product_type: RIFG_RUNW_GUNW`) yields multiple products.

In [ ]:
# Papermill parameters. Override at run time with `papermill -p ...`.
runconfig_s3 = ""          # inline runconfig YAML text (containing s3:// links), or a local path
netrc_content = ""         # netrc credentials text (machine/login/password)
output_dir = "output"      # SAS product output directory
scratch_dir = "scratch"    # SAS scratch directory
localized_runconfig = "runconfig_localized.yaml"  # written localized runconfig

# hysds specifications
_time_limit = 172800
_soft_time_limit = 172800
_disk_usage = '200GB'
_submission_type = 'iteration'
_label = 'ISCE3 PGE'

In [ ]:
import os
import sys
import stat
import shutil
import subprocess
from pathlib import Path

import yaml
import h5py

# Fail loud if we are not running under the isce3_src kernel: nisar must import
# in-process here, and the localizer's `stage_dem` subprocess uses sys.executable.
import nisar  # noqa: F401
print(f"interpreter: {sys.executable}")

# Make nisar_products importable. It is pip-installed into isce3_src in the image;
# fall back to the source checkout for local/dev runs.
try:
    import nisar_products  # noqa: F401
except ImportError:
    src = os.environ.get("NISAR_ONDEMAND_SRC", "/home/jovyan/ondemand-resources/src")
    if src not in sys.path:
        sys.path.insert(0, src)
    import nisar_products  # noqa: F401

from nisar_products import configure
from nisar_products.runconfig_localizer import localize_runconfig

# Track/frame GeoPackage shipped with this repo. Constant (not a papermill param):
# it is a fixed reference DB used only by the INSAR DEM-bbox fallback.
TRACKFRAME_DB = "/home/jovyan/nisar-isce3-pge/data/NISAR_TrackFrame_L_20260618.gpkg"

# Write netrc so earthaccess can fetch temporary DAAC S3 credentials.
if netrc_content.strip():
    netrc_path = Path.home() / ".netrc"
    netrc_path.write_text(netrc_content if netrc_content.endswith("\n") else netrc_content + "\n")
    netrc_path.chmod(stat.S_IRUSR | stat.S_IWUSR)  # 0600, required by netrc consumers
    print(f"wrote {netrc_path}")

# Point the track/frame lookup at this repo's GeoPackage.
configure(trackframe_db=TRACKFRAME_DB)
print(f"trackframe_db: {TRACKFRAME_DB}")

In [ ]:
# The runconfig is delivered inline as YAML text (a `destination: context` param,
# so HySDS stores the submitted value verbatim). Materialize it to a local file
# for the localizer, which reads its argument with a plain open(). A bare local
# path is still accepted for back-compat / local dev runs.
if runconfig_s3 and os.path.isfile(runconfig_s3):
    rc_input = runconfig_s3
    print(f"using runconfig path: {rc_input}")
else:
    if not runconfig_s3.strip():
        raise ValueError("runconfig_s3 parameter is empty")

    rc_input = "runconfig_input.yaml"
    Path(rc_input).write_text(runconfig_s3)
    print(f"wrote inline runconfig to {rc_input} ({len(runconfig_s3)} bytes)")

# Download every s3:// input in the runconfig, stage the DEM, rewrite paths to
# local absolute paths, and pin the output/scratch dirs. Returns the local YAML path.
local_rc = localize_runconfig(
    rc_input,
    localized_runconfig,
    output_dir=output_dir,
    scratch_dir=scratch_dir,
)
print(f"localized runconfig: {local_rc}")

In [ ]:
# Resolve workflow module + the list of deliverable HDF5 files from the localized runconfig.
with open(local_rc) as f:
    cfg = yaml.safe_load(f)
groups = cfg["runconfig"]["groups"]
product_type = groups["primary_executable"]["product_type"]
sas_output_file = groups["product_path_group"]["sas_output_file"]

# product_type -> nisar.workflows module. The InSAR family is one umbrella module and its
# product_type may be a single code (RIFG) or an underscore-joined combo (RIFG_RUNW_GUNW).
SINGLE_MODULE = {"RSLC": "focus", "GSLC": "gslc", "GCOV": "gcov", "SME2": "sme2"}
INSAR_CODES = {"RIFG", "RUNW", "ROFF", "GUNW", "GOFF"}

if product_type in SINGLE_MODULE:
    module = SINGLE_MODULE[product_type]
elif set(product_type.split("_")) <= INSAR_CODES:
    module = "insar"
else:
    raise ValueError(f"unsupported product_type {product_type!r}")

# Enumerate the HDF5 files the workflow will emit. For InSAR, reuse the SAS's own mapping
# (h5_prep.get_products_and_paths) so combined types like RIFG_RUNW_GUNW yield every
# co-product (output/RIFG_product.h5, output/RUNW_product.h5, output/GUNW_product.h5).
# Intermediates written to the scratch dir are NOT deliverables.
if module == "insar":
    from nisar.workflows.h5_prep import get_products_and_paths
    _subprods, h5_paths = get_products_and_paths(groups)
    scratch_dir_abs = os.path.abspath(groups["product_path_group"]["scratch_path"])
    deliverables = [
        p for p in h5_paths.values()
        if os.path.abspath(os.path.dirname(p)) != scratch_dir_abs
    ]
else:
    deliverables = [sas_output_file]

print(f"product_type={product_type} module=nisar.workflows.{module}")
print("deliverables:", deliverables)

In [ ]:
# Run the SAS workflow in the isce3_src conda env so its activation scripts set
# PYTHONPATH/LD_LIBRARY_PATH for libisce3.
#
# We WANT `conda run -n isce3_src` to work (portable, also correct for local dev
# where the env prefix differs). `-n` resolves the NAME against conda's envs_dirs,
# which the Dockerfile now registers (conda config --append envs_dirs
# /opt/conda/envs). But envs_dirs is a runtime property (conda root + HOME +
# condarc), so verify it actually took here rather than assume: probe `-n` first,
# log the outcome, and fall back to the known absolute prefix if it did not.
ISCE3_ENV_NAME = "isce3_src"
ISCE3_ENV_PREFIX = "/opt/conda/envs/isce3_src"  # Dockerfile: --envs-dir/--env-name

# Cheap resolution check: does `conda run -n isce3_src` find the env at all?
probe = subprocess.run(
    ["conda", "run", "-n", ISCE3_ENV_NAME, "python", "-c", "import sys; print(sys.prefix)"],
    capture_output=True, text=True,
)
if probe.returncode == 0:
    conda_target = ["-n", ISCE3_ENV_NAME]
    print(f"[env] `-n {ISCE3_ENV_NAME}` RESOLVES -> {probe.stdout.strip()} "
          f"(name resolution fixed)", flush=True)
else:
    conda_target = ["-p", ISCE3_ENV_PREFIX]
    print(f"[env] `-n {ISCE3_ENV_NAME}` FAILED to resolve; falling back to "
          f"`-p {ISCE3_ENV_PREFIX}`. conda said:\n{probe.stderr.strip()}", flush=True)

# --live-stream forwards stdout/stderr in real time instead of buffering.
cmd = [
    "conda", "run", *conda_target, "--live-stream",
    "python", "-m", f"nisar.workflows.{module}", local_rc,
]
print("running:", " ".join(cmd), flush=True)
subprocess.run(cmd, check=True)
print("SAS workflow completed")

In [ ]:
# Stage out each deliverable. The SAS writes the fully-substituted granule name into each
# product HDF5. For every deliverable: read its granuleId, create a sibling directory (next
# to output/) named after the granule, MOVE the HDF5 into it as <granuleId>.h5, and drop a
# copy of the localized runconfig used to produce it.
stage_root = os.path.dirname(os.path.abspath(output_dir))  # parallel to output/

staged = []
for src in deliverables:
    if not os.path.exists(src):
        raise FileNotFoundError(f"expected SAS output not found: {src}")

    with h5py.File(src, "r") as h5:
        granule_id = h5["/science/LSAR/identification/granuleId"][()]
    granule_id = granule_id.decode() if isinstance(granule_id, bytes) else str(granule_id)
    granule_id = granule_id.strip()
    if "{" in granule_id or "}" in granule_id:
        raise ValueError(f"granuleId still has unfilled tokens: {granule_id!r} (from {src})")

    granule = granule_id[:-3] if granule_id.endswith(".h5") else granule_id
    dest_dir = os.path.join(stage_root, granule)
    os.makedirs(dest_dir, exist_ok=True)

    dest_h5 = os.path.join(dest_dir, granule + ".h5")
    shutil.move(src, dest_h5)
    shutil.copy2(local_rc, os.path.join(dest_dir, "runconfig.yaml"))
    staged.append(dest_dir)
    print(f"staged {src} -> {dest_h5}")

print(f"\n{len(staged)} product(s) staged:")
for d in staged:
    print(" ", d)